# Sistema de apoio à classificação de câncer de mama

**Tech Challenge — Fase 1 | Python 3.12**

Este notebook apresenta um fluxo completo e organizado: valida primeiro se a base é compatível com o problema de câncer de mama; depois explora, prepara, treina, avalia e explica os modelos.

> **Uso responsável:** esta é uma solução educacional de apoio à triagem. Ela não confirma nem descarta câncer e não substitui a avaliação de profissionais de saúde.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_utils import load_raw_data

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2.1 Carregamento, escopo e validação da base

Esta é a primeira etapa executável do notebook. Ela carrega a base, padroniza os rótulos de diagnóstico e prepara `data`, `X` e `y` para as etapas seguintes. Logo depois, o notebook confirma se as colunas são compatíveis com o problema de câncer de mama antes de iniciar a EDA ou o treinamento.

In [ ]:
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_dir / "data" / "data.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {data_path}. Baixe o data.csv seguindo data/README.md."
    )

raw_data = pd.read_csv(data_path)
data = raw_data.dropna(axis=1, how="all").copy()
data = data.drop(columns=["id"], errors="ignore")

if "diagnosis" not in data.columns:
    raise ValueError("A base precisa conter a coluna 'diagnosis' com os rótulos M e B. Não identificado como cancer de mama.")

# Normaliza textos: remove espaços e converte para letras minúsculas
data["diagnosis"] = (
    data["diagnosis"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Aceita formatos diferentes para benigno e maligno
label_map = {
    "m": "M",
    "malignant": "M",
    "maligno": "M",
    "b": "B",
    "benign": "B",
    "benigno": "B",
}

found_labels = set(data["diagnosis"].dropna().unique())
invalid_labels = found_labels - set(label_map.keys())

if invalid_labels:
    raise ValueError(
        f"Rótulos inesperados em diagnosis: {invalid_labels}. "
        "Use M, B, malignant, benign, maligno ou benigno."
    )

# Padroniza os rótulos para M e B
data["diagnosis"] = data["diagnosis"].map(label_map)

TARGET = "diagnosis"
y = data[TARGET].map({"M": 1, "B": 0})
X = data.drop(columns=TARGET)

print(f"Amostras: {len(data)} | Atributos candidatos: {X.shape[1]}")
print("\nDistribuição do desfecho (1 = maligno):")
display(y.value_counts().rename(index={0: "Benigno", 1: "Maligno"}).to_frame("quantidade"))
print("\nValores ausentes por coluna (somente colunas com ausência):")
display(X.isna().sum().loc[lambda s: s.gt(0)].sort_values(ascending=False))

## 2.2 Status de compatibilidade com câncer de mama

Após a normalização dos rótulos, esta etapa verifica se o CSV possui medidas morfológicas esperadas pelo dataset Breast Cancer Wisconsin. Se as colunas não existirem, o notebook mostra o status de incompatibilidade e encerra antes da EDA, do treinamento e da previsão.

In [ ]:
expected_breast_features = {
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "concavity_mean", "radius_worst", "area_worst",
}
missing_features = sorted(expected_breast_features - set(data.columns))

if missing_features:
    identificacao_csv = {
        "status": "CSV incompatível",
        "dominio_identificado": "Não compatível com câncer de mama",
        "mensagem": "Faltam medidas esperadas pelo Breast Cancer Wisconsin.",
        "campos_ausentes": ", ".join(missing_features),
    }
    display(pd.DataFrame([identificacao_csv]))
    raise ValueError(
        "Base incompatível com câncer de mama. "
        "Nenhum treinamento ou diagnóstico foi realizado."
    )

df = data.copy()
identificacao_csv = {
    "status": "CSV compatível",
    "dominio_identificado": "Câncer de mama",
    "mensagem": "Campos e rótulos compatíveis com o modelo Breast Cancer Wisconsin.",
    "amostras": int(len(data)),
    "atributos": int(X.shape[1]),
}
display(pd.DataFrame([identificacao_csv]))
df.head()

## 3. Análise Exploratória de Dados (EDA)

### 3.1 Visualização dos dados validados

Com o CSV confirmado como compatível, as primeiras linhas são exibidas para conhecer as colunas que serão exploradas.

In [ ]:
df.info()

### 3.2 Estatísticas descritivas

Resume os atributos numéricos com média, desvio padrão, mínimo, máximo e percentis para identificar escalas, dispersão e valores possivelmente atípicos.

In [ ]:
df.describe()

In [ ]:
# Checar valores ausentes
df.isnull().sum().sort_values(ascending=False)

### 3.3 Distribuição da variável-alvo (diagnóstico)

Compara a quantidade de casos benignos e malignos. Essa etapa identifica desbalanceamento entre classes antes do treinamento.

In [ ]:
target_col = "diagnosis"  # ajustar conforme o nome real da coluna no dataset

sns.countplot(data=df, x=target_col)
plt.title("Distribuição de diagnósticos (M = maligno, B = benigno)")
plt.show()

### 3.4 Análise de correlação

Avalia a associação entre os atributos numéricos e o diagnóstico. Correlação ajuda a explorar padrões, mas não demonstra causa clínica.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Matriz de correlação")
plt.show()

## 4. Modelagem, avaliação e serviço de inferência

Após a validação e a EDA, as próximas etapas preparam os dados, treinam os modelos, avaliam o desempenho e disponibilizam uma função de apoio à triagem.

### 4.1 Preparação do ambiente e definição do problema

O modelo classifica amostras como **malignas** ou **benignas** usando atributos numéricos do dataset público Breast Cancer Wisconsin (Diagnostic). Esses atributos foram extraídos de imagens digitalizadas de massas mamárias; eles não são mamografias brutas.

> **Limite clínico:** este projeto é educacional e de apoio à decisão. Não é um dispositivo médico, não substitui exame clínico, laudo ou a decisão final de profissionais de saúde.

In [ ]:
# Ambiente previsto: Python 3.12 e dependências de requirements.txt
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
print(f"Python em execução: {sys.version.split()[0]}")

### 4.2 Preparação do alvo e atributos

A validação do escopo já foi concluída na seção 2. Nesta etapa, o diagnóstico padronizado é convertido para um alvo numérico: `M` vira `1` (maligno) e `B` vira `0` (benigno). Os demais campos se tornam os atributos usados pelos modelos.

In [ ]:
data = df.copy()
TARGET = "diagnosis"
y = data[TARGET].map({"M": 1, "B": 0})
X = data.drop(columns=TARGET)

if y.isna().any():
    raise ValueError("Não foi possível converter todos os rótulos de diagnóstico.")

print(f"Amostras: {len(data)} | Atributos candidatos: {X.shape[1]}")
print("\nDistribuição do desfecho (1 = maligno):")
display(y.value_counts().rename(index={0: "Benigno", 1: "Maligno"}).to_frame("quantidade"))
print("\nValores ausentes por coluna (somente colunas com ausência):")
display(X.isna().sum().loc[lambda s: s.gt(0)].sort_values(ascending=False))

### 4.3 Padrões relevantes para modelagem

A proporção de classes é importante porque, em triagem, deixar de identificar um caso maligno (falso negativo) pode ser mais grave do que encaminhar um caso benigno para investigação adicional. A matriz abaixo ajuda a identificar atributos redundantes e possíveis padrões associados ao desfecho.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(x=data[TARGET], order=["B", "M"], ax=axes[0], hue=data[TARGET], legend=False)
axes[0].set(title="Distribuição do diagnóstico", xlabel="Diagnóstico", ylabel="Número de amostras")

correlations = pd.concat([X, y.rename("maligno")], axis=1).corr(numeric_only=True)["maligno"].drop("maligno")
top_correlations = correlations.abs().sort_values(ascending=False).head(10).index
sns.barplot(
    x=correlations.loc[top_correlations].sort_values(),
    y=correlations.loc[top_correlations].sort_values().index,
    ax=axes[1],
    color="#b2182b",
)
axes[1].set(title="Top 10 correlações com malignidade", xlabel="Correlação de Pearson", ylabel="Atributo")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(X.loc[:, top_correlations].corr(), cmap="vlag", center=0, square=True)
plt.title("Correlação entre os atributos mais associados ao desfecho")
plt.show()

print("Correlação é associação linear, não causalidade. Atributos altamente correlacionados podem representar a mesma característica clínica.")

### 4.4 Pré-processamento e separação treino/teste

A divisão estratificada reserva 20% dos dados para teste, sem exposição durante o treinamento. O `Pipeline` evita vazamento de dados: imputação, escala e codificação são ajustadas apenas nos dados de treino. Embora esta base seja numérica, o transformador também trata colunas categóricas para tornar o serviço reutilizável.

In [ ]:
# ==========================================================
# VALIDAÇÃO DAS CLASSES E PRÉ-PROCESSAMENTO ADAPTÁVEL
# ==========================================================
 
class_counts = y.value_counts()
n_classes = y.nunique()
total_amostras = len(X)
 
# Indica se há dados suficientes para treinamento supervisionado
modelagem_habilitada = (
    n_classes >= 2
    and class_counts.min() >= 2
)
 
if not modelagem_habilitada:
    motivo_modelagem_indisponivel = (
        "Modelagem não executada. A base possui apenas uma classe ou não tem "
        "amostras suficientes em cada classe para divisão estratificada. "
        f"Distribuição encontrada: {class_counts.to_dict()}."
    )
 
    print("=" * 70)
    print("STATUS: DADOS INSUFICIENTES PARA TREINAMENTO")
    print(motivo_modelagem_indisponivel)
    print("A análise exploratória pode continuar, mas não é possível treinar")
    print("um classificador entre benigno e maligno.")
    print("=" * 70)
 
    # Mantém variáveis definidas para evitar erro em células posteriores
    X_train = X.copy()
    X_test = X.iloc[0:0].copy()
 
    y_train = y.copy()
    y_test = y.iloc[0:0].copy()
 
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
 
    preprocessor = None
 
else:
    motivo_modelagem_indisponivel = None
 
    # Calcula tamanho de teste adaptável
    test_size_padrao = int(np.ceil(total_amostras * 0.20))
 
    # Garante pelo menos uma amostra de cada classe no teste
    test_size = max(test_size_padrao, n_classes)
 
    # Garante pelo menos uma amostra de cada classe no treino
    max_test_size = total_amostras - n_classes
 
    if test_size > max_test_size:
        test_size = max_test_size
 
    # Divisão estratificada
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=RANDOM_STATE,
    )
 
    numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()
 
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
 
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
 
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
 
    print("=" * 70)
    print("STATUS: DADOS VÁLIDOS PARA TREINAMENTO")
    print(f"Distribuição original: {class_counts.to_dict()}")
    print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
    print(
        f"Numéricas: {len(numeric_features)} | "
        f"Categóricas: {len(categorical_features)}"
    )
    print("=" * 70)

### 4.5 Modelagem

Serão comparados dois modelos: **Regressão Logística** (baseline interpretável) e **Random Forest** (captura relações não lineares). Ambos recebem pesos balanceados para reduzir o impacto da classe majoritária durante o ajuste.

In [ ]:
models = {
    "Regressão Logística": LogisticRegression(
        max_iter=2_000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

fitted_models = {}
results = []
for name, estimator in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipeline
    results.append({
        "modelo": name,
        "accuracy": accuracy_score(y_test, predictions),
        "recall_maligno": recall_score(y_test, predictions, pos_label=1),
        "f1_maligno": f1_score(y_test, predictions, pos_label=1),
        "roc_auc": roc_auc_score(y_test, probabilities),
    })

results_df = pd.DataFrame(results).sort_values(
    ["recall_maligno", "f1_maligno", "roc_auc"], ascending=False
).reset_index(drop=True)
display(results_df.style.format({column: "{:.3f}" for column in results_df.columns[1:]}))

best_name = results_df.loc[0, "modelo"]
best_model = fitted_models[best_name]
print(f"Modelo selecionado pelo recall de malignidade: {best_name}")

### 4.6 Avaliação do modelo selecionado

O modelo é avaliado somente no conjunto de teste com relatório de classificação, matriz de confusão e curva ROC. O recall da classe maligna é priorizado, mas deve ser interpretado junto com F1-score, AUC e falsos positivos.

In [ ]:
test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, test_predictions, target_names=["Benigno", "Maligno"], digits=3))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, display_labels=["Benigno", "Maligno"], cmap="Blues", ax=axes[0]
)
axes[0].set_title(f"Matriz de confusão — {best_name}")
RocCurveDisplay.from_predictions(y_test, test_probabilities, name=best_name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Curva ROC no conjunto de teste")
plt.tight_layout()
plt.show()

print("Métrica prioritária: recall da classe maligna. Ainda assim, ele deve ser analisado junto a precisão, F1 e falsos positivos para evitar encaminhamentos desnecessários.")

### 4.7 Explicabilidade

A importância por permutação mede quanto a métrica cai quando os valores de cada atributo são embaralhados no teste. Ela é agnóstica ao modelo e mostra associação preditiva, não causalidade. Para uma explicação local, a célula seguinte usa SHAP quando a biblioteca estiver instalada.

In [ ]:
permutation = permutation_importance(
    best_model, X_test, y_test, scoring="recall", n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1
)
importance_df = pd.DataFrame({
    "atributo": X_test.columns,
    "importancia_media": permutation.importances_mean,
    "desvio_padrao": permutation.importances_std,
}).sort_values("importancia_media", ascending=False).head(12)

display(importance_df)
plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df.sort_values("importancia_media"), x="importancia_media", y="atributo", color="#2166ac")
plt.title(f"Importância por permutação — {best_name}")
plt.xlabel("Queda média do recall após permutação")
plt.ylabel("Atributo")
plt.show()

try:
    import shap

    transformed_train = best_model.named_steps["preprocessor"].transform(X_train)
    transformed_test = best_model.named_steps["preprocessor"].transform(X_test)
    feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    estimator = best_model.named_steps["model"]

    explainer = shap.Explainer(estimator, transformed_train, feature_names=feature_names)
    shap_values = explainer(transformed_test)

    # Modelos binários podem retornar valores SHAP com ou sem dimensão de classe.
    values_for_malignancy = shap_values[..., 1] if shap_values.values.ndim == 3 else shap_values
    shap.plots.bar(values_for_malignancy, max_display=12)
except ImportError:
    print("SHAP não está instalado. Execute `pip install -r requirements.txt` para gerar esta visualização.")
except Exception as error:
    print(f"Não foi possível gerar o gráfico SHAP nesta execução: {error}")

### 4.8 Serviço de inferência e persistência

A função abaixo recebe um `DataFrame` com as mesmas colunas de entrada do treinamento e devolve a probabilidade de malignidade. Se os campos não forem compatíveis, ela não faz previsão e informa que o arquivo não pode ser avaliado por este modelo de câncer de mama. Dados incompatíveis não permitem concluir que a pessoa não tem câncer de mama.

O limiar padrão é 0,50 apenas como referência técnica; em uma validação clínica ele deve ser definido com especialistas, considerando sensibilidade, capacidade de encaminhamento e custos de erro.

In [ ]:
def prever_risco_cancer_mama(amostras: pd.DataFrame, limiar: float = 0.50) -> pd.DataFrame:
    """Gera apoio de triagem; não fornece diagnóstico clínico definitivo."""
    if not 0 < limiar < 1:
        raise ValueError("O limiar deve estar entre 0 e 1.")

    missing_columns = sorted(set(X.columns) - set(amostras.columns))
    if missing_columns:
        return pd.DataFrame({
            "status": ["Entrada incompatível"],
            "mensagem": [
                "Este arquivo não possui os campos esperados pelo modelo de câncer de mama. "
                "Nenhuma previsão foi realizada; isso não confirma nem descarta câncer de mama."
            ],
            "colunas_ausentes": [", ".join(missing_columns)],
        })

    input_data = amostras.loc[:, X.columns].copy()
    probability = best_model.predict_proba(input_data)[:, 1]
    triage = np.where(probability >= limiar, "Encaminhar para avaliação médica", "Sem alerta pelo limiar técnico")
    return pd.DataFrame({
        "probabilidade_malignidade": probability,
        "limiar": limiar,
        "orientacao": triage,
        "aviso": "Resultado de apoio; profissional de saúde deve avaliar o caso.",
    }, index=amostras.index)

model_dir = project_dir / "models"
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "modelo_cancer_mama.joblib"
joblib.dump({"model": best_model, "features": X.columns.tolist(), "threshold": 0.50}, model_path)
print(f"Artefato salvo em: {model_path}")

## 5. Testes manuais e geração de artefatos

### 5.1 Início dos testes — insira novos dados aqui

**Os testes começam na célula abaixo.** Por padrão, ela usa três exemplos reservados no conjunto de teste. Para testar um CSV novo, salve-o em `data/nova_amostra.csv`, altere `USAR_CSV_NOVO` para `True` e mantenha as mesmas colunas de atributos usadas no treinamento.

In [ ]:
# ===== INÍCIO DOS TESTES MANUAIS =====
# False: testa exemplos reservados pelo próprio notebook.
# True: lê data/nova_amostra.csv para testar novas informações.
USAR_CSV_NOVO = False

if USAR_CSV_NOVO:
    novo_csv_path = project_dir / "data" / "data.csv"
    if not novo_csv_path.exists():
        raise FileNotFoundError(
            f"Arquivo de teste não encontrado: {novo_csv_path}. "
            "Crie-o com as mesmas colunas de atributos da base de treinamento."
        )
    amostras_para_teste = pd.read_csv(novo_csv_path).drop(
        columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore"
    )
    origem_teste = f"CSV novo: {novo_csv_path.name}"
else:
    amostras_para_teste = X_test.head(3).copy()
    origem_teste = "Três exemplos do conjunto de teste"

print(f"Origem dos testes: {origem_teste}")
resultado_testes = prever_risco_cancer_mama(amostras_para_teste)
display(resultado_testes)

# O resultado é apoio educacional à triagem, não um diagnóstico clínico definitivo.

### 5.2 Gerar análise diagnóstica em PDF

Depois de executar o notebook, a célula abaixo registra o status do CSV, a distribuição dos diagnósticos e as métricas do modelo. Em seguida, gera `reports/analise_diagnostica_cancer_mama.pdf` com essas informações.

In [ ]:
import sys
import subprocess
from pathlib import Path
from xml.sax.saxutils import escape

# Instala ReportLab caso necessário
try:
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import cm
    from reportlab.platypus import (
        Paragraph,
        SimpleDocTemplate,
        Spacer,
        Table,
        TableStyle,
        Image,
        PageBreak,
    )
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab"])

    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import cm
    from reportlab.platypus import (
        Paragraph,
        SimpleDocTemplate,
        Spacer,
        Table,
        TableStyle,
        Image,
        PageBreak,
    )

from sklearn.metrics import ConfusionMatrixDisplay

# ==========================================================
# PASTAS E ARQUIVOS
# ==========================================================

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

reports_dir = project_dir / "reports"
assets_dir = reports_dir / "pdf_assets"

reports_dir.mkdir(exist_ok=True)
assets_dir.mkdir(exist_ok=True)

pdf_path = reports_dir / "analise_diagnostica_cancer_mama.pdf"

# Repositório oficial do projeto (mesmo link usado no PDF gerado pelo frontend)
REPOSITORIO_URL = "https://github.com/Gusta2150/desafio_tech"

# ==========================================================
# INFORMAÇÕES DA EXECUÇÃO
# ==========================================================

identificacao_csv = globals().get(
    "identificacao_csv",
    {
        "status": "Não executado",
        "dominio_identificado": "Não informado",
        "mensagem": "A validação do CSV não foi executada nesta sessão.",
    },
)

quantidade_benigno = int((y == 0).sum()) if "y" in globals() else 0
quantidade_maligno = int((y == 1).sum()) if "y" in globals() else 0

modelo_selecionado = globals().get("best_name", "Modelo não treinado")

origem_teste = globals().get(
    "origem_teste",
    "Testes manuais não executados",
)

metricas_modelo = {}

if "results_df" in globals() and not results_df.empty:
    metricas_modelo = {
        key: float(value)
        for key, value in results_df.iloc[0].drop(labels="modelo").items()
    }

# Descrição adaptável da divisão treino/teste
if "X_train" in globals() and "X_test" in globals():
    total_divisao = len(X_train) + len(X_test)

    percentual_treino = (len(X_train) / total_divisao) * 100
    percentual_teste = (len(X_test) / total_divisao) * 100

    descricao_divisao = (
        f"Nesta execução foram utilizadas {len(X_train)} amostras para treino "
        f"({percentual_treino:.1f}%) e {len(X_test)} amostras para teste "
        f"({percentual_teste:.1f}%)."
    )
else:
    descricao_divisao = (
        "A divisão entre treino e teste não foi executada nesta sessão."
    )

# ==========================================================
# ESTILOS DO PDF
# ==========================================================

styles = getSampleStyleSheet()

titulo_style = ParagraphStyle(
    "Titulo",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=19,
    leading=23,
    alignment=TA_CENTER,
    spaceAfter=16,
)

subtitulo_style = ParagraphStyle(
    "Subtitulo",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=13,
    leading=16,
    spaceBefore=10,
    spaceAfter=7,
)

texto_style = ParagraphStyle(
    "Texto",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=10,
    leading=14,
    spaceAfter=8,
)

link_style = ParagraphStyle(
    "Link",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=10,
    leading=14,
    spaceAfter=8,
    textColor=colors.HexColor("#1f4e78"),
)

def texto_pdf(valor, estilo=texto_style):
    return Paragraph(escape(str(valor)), estilo)

def criar_tabela(linhas, larguras=(6.2 * cm, 10.2 * cm)):
    tabela = Table(linhas, colWidths=list(larguras))

    tabela.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1f4e78")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#9ca3af")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("BACKGROUND", (0, 1), (-1, -1), colors.HexColor("#f8fafc")),
                ("LEFTPADDING", (0, 0), (-1, -1), 6),
                ("RIGHTPADDING", (0, 0), (-1, -1), 6),
                ("TOPPADDING", (0, 0), (-1, -1), 5),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ]
        )
    )

    return tabela

def adicionar_numero_pagina(canvas, document):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.HexColor("#4b5563"))

    canvas.drawRightString(
        A4[0] - 1.5 * cm,
        1.0 * cm,
        f"Página {document.page}",
    )

    canvas.restoreState()

# ==========================================================
# GERAÇÃO DOS GRÁFICOS
# ==========================================================

graficos = []

# Gráfico 1: distribuição entre benignos e malignos
if "y" in globals():
    grafico_distribuicao = assets_dir / "distribuicao_diagnosticos.png"

    plt.figure(figsize=(8, 5))

    sns.barplot(
        x=["Benigno", "Maligno"],
        y=[quantidade_benigno, quantidade_maligno],
        color="#1f77b4",
    )

    plt.title("Distribuição dos diagnósticos")
    plt.xlabel("Diagnóstico")
    plt.ylabel("Quantidade de amostras")
    plt.tight_layout()

    plt.savefig(grafico_distribuicao, dpi=160)
    plt.close()

    graficos.append((
        "Distribuição dos diagnósticos",
        grafico_distribuicao,
        "O gráfico mostra a quantidade de casos benignos e malignos presentes na base."
    ))

# Gráfico 2: correlação com malignidade
if "X" in globals() and "y" in globals():
    grafico_correlacao = assets_dir / "correlacao_malignidade.png"

    correlacoes = (
        pd.concat([X, y.rename("maligno")], axis=1)
        .corr(numeric_only=True)["maligno"]
        .drop("maligno")
    )

    top_atributos = correlacoes.abs().sort_values(ascending=False).head(10).index
    dados_correlacao = correlacoes.loc[top_atributos].sort_values()

    plt.figure(figsize=(9, 6))

    sns.barplot(
        x=dados_correlacao.values,
        y=dados_correlacao.index,
        color="#b2182b",
    )

    plt.title("Top 10 correlações com malignidade")
    plt.xlabel("Correlação de Pearson")
    plt.ylabel("Atributo")
    plt.tight_layout()

    plt.savefig(grafico_correlacao, dpi=160)
    plt.close()

    graficos.append((
        "Correlação com malignidade",
        grafico_correlacao,
        "A correlação mostra associação estatística com o diagnóstico, mas não demonstra causa clínica."
    ))

# Gráfico 3: matriz de confusão
if (
    "best_model" in globals()
    and "X_test" in globals()
    and "y_test" in globals()
):
    grafico_matriz = assets_dir / "matriz_confusao.png"

    previsoes = best_model.predict(X_test)

    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        previsoes,
        display_labels=["Benigno", "Maligno"],
        cmap="Blues",
        ax=ax,
    )

    ax.set_title(f"Matriz de confusão — {modelo_selecionado}")

    plt.tight_layout()
    plt.savefig(grafico_matriz, dpi=160)
    plt.close()

    graficos.append((
        "Matriz de confusão",
        grafico_matriz,
        "A matriz mostra os acertos e erros do modelo, incluindo falsos positivos e falsos negativos."
    ))

# Gráfico 4: importância dos atributos
if "importance_df" in globals() and not importance_df.empty:
    grafico_importancia = assets_dir / "importancia_atributos.png"

    dados_importancia = importance_df.sort_values("importancia_media")

    plt.figure(figsize=(9, 6))

    sns.barplot(
        data=dados_importancia,
        x="importancia_media",
        y="atributo",
        color="#2166ac",
    )

    plt.title("Importância dos atributos")
    plt.xlabel("Impacto médio no desempenho")
    plt.ylabel("Atributo")
    plt.tight_layout()

    plt.savefig(grafico_importancia, dpi=160)
    plt.close()

    graficos.append((
        "Importância por permutação",
        grafico_importancia,
        "Este gráfico mostra quais atributos mais impactaram o desempenho do modelo."
    ))

# ==========================================================
# CRIAÇÃO DO PDF
# ==========================================================

documento = SimpleDocTemplate(
    str(pdf_path),
    pagesize=A4,
    leftMargin=1.5 * cm,
    rightMargin=1.5 * cm,
    topMargin=1.5 * cm,
    bottomMargin=1.5 * cm,
)

conteudo = []

# Título
conteudo.append(
    texto_pdf(
        "Análise Diagnóstica de Câncer de Mama",
        titulo_style,
    )
)

# Integrantes
conteudo.append(texto_pdf("Integrantes", subtitulo_style))
conteudo.append(texto_pdf("• Gustavo Leite"))
conteudo.append(texto_pdf("• Luiz Fellipe"))
conteudo.append(texto_pdf("• Gabriel Wesley"))
conteudo.append(
    Paragraph(
        f'Repositório: <link href="{escape(REPOSITORIO_URL)}">{escape(REPOSITORIO_URL)}</link>',
        link_style,
    )
)
conteudo.append(Spacer(1, 8))

# Identificação do CSV
conteudo.append(texto_pdf("Identificação do CSV", subtitulo_style))

tabela_identificacao = [
    [texto_pdf("Campo"), texto_pdf("Resultado")],
    [
        texto_pdf("Status"),
        texto_pdf(identificacao_csv.get("status", "Não informado")),
    ],
    [
        texto_pdf("Domínio identificado"),
        texto_pdf(
            identificacao_csv.get(
                "dominio_identificado",
                "Não informado",
            )
        ),
    ],
    [
        texto_pdf("Mensagem"),
        texto_pdf(identificacao_csv.get("mensagem", "Não informado")),
    ],
]

conteudo.append(criar_tabela(tabela_identificacao))
conteudo.append(Spacer(1, 10))

# Discussão da análise exploratória
conteudo.append(
    texto_pdf(
        "Discussão da análise exploratória",
        subtitulo_style,
    )
)

conteudo.append(
    texto_pdf(
        f"A base analisada possui {len(data)} amostras e {X.shape[1]} atributos "
        "utilizados na classificação. Foram analisados tipos de dados, valores "
        "ausentes, estatísticas descritivas, distribuição das classes e correlação "
        "entre características morfológicas e malignidade."
    )
)

conteudo.append(
    texto_pdf(
        "A análise exploratória é importante porque permite identificar problemas "
        "nos dados antes do treinamento, como valores ausentes, escalas diferentes "
        "entre atributos e possível desbalanceamento entre diagnósticos."
    )
)

# Distribuição dos diagnósticos
conteudo.append(
    texto_pdf(
        "Distribuição dos diagnósticos",
        subtitulo_style,
    )
)

tabela_distribuicao = [
    [texto_pdf("Classe"), texto_pdf("Quantidade")],
    [texto_pdf("Benigno"), texto_pdf(quantidade_benigno)],
    [texto_pdf("Maligno"), texto_pdf(quantidade_maligno)],
]

conteudo.append(criar_tabela(tabela_distribuicao))
conteudo.append(Spacer(1, 10))

# Estratégias de pré-processamento
conteudo.append(
    texto_pdf(
        "Estratégias de pré-processamento",
        subtitulo_style,
    )
)

conteudo.append(
    texto_pdf(
        "O pré-processamento remove identificadores sem valor preditivo, elimina "
        "colunas totalmente vazias, trata valores ausentes com mediana para campos "
        "numéricos e valor mais frequente para campos categóricos. As variáveis "
        "numéricas são normalizadas com StandardScaler e variáveis categóricas, "
        "quando existirem, são convertidas com OneHotEncoder."
    )
)

conteudo.append(
    texto_pdf(
        "Os dados são separados de forma estratificada entre treino e teste. "
        "A proporção padrão é de aproximadamente 80% para treino e 20% para teste. "
        "Entretanto, quando a base possui poucas amostras, o sistema adapta "
        "automaticamente o tamanho do conjunto de teste para garantir que exista "
        "pelo menos um caso benigno e um caso maligno nos dois grupos."
    )
)

conteudo.append(texto_pdf(descricao_divisao))

# Modelos utilizados
conteudo.append(
    texto_pdf(
        "Modelos utilizados e justificativa",
        subtitulo_style,
    )
)

conteudo.append(
    texto_pdf(
        "Foram utilizados dois algoritmos. A Regressão Logística foi escolhida "
        "por ser um modelo mais simples, interpretável e capaz de gerar "
        "probabilidades. O Random Forest foi escolhido por capturar relações "
        "não lineares e interações mais complexas entre os atributos."
    )
)

conteudo.append(
    texto_pdf(
        f"O modelo selecionado nesta execução foi: {modelo_selecionado}."
    )
)

# Resultados do modelo
conteudo.append(
    texto_pdf(
        "Resultados do modelo",
        subtitulo_style,
    )
)

if metricas_modelo:
    tabela_metricas = [
        [texto_pdf("Métrica"), texto_pdf("Valor")]
    ]

    for metrica, valor in metricas_modelo.items():
        tabela_metricas.append([
            texto_pdf(metrica),
            texto_pdf(f"{valor:.3f}"),
        ])

    conteudo.append(criar_tabela(tabela_metricas))

else:
    conteudo.append(
        texto_pdf(
            "As métricas não estão disponíveis. Execute as células de modelagem "
            "antes de gerar o PDF."
        )
    )

conteudo.append(
    texto_pdf(
        "O recall de malignidade é uma métrica prioritária porque indica quantos "
        "casos malignos presentes foram identificados pelo modelo. Accuracy, "
        "F1-score e ROC-AUC são analisados em conjunto para evitar interpretações isoladas."
    )
)

# Interpretação dinâmica dos resultados
conteudo.append(
    texto_pdf(
        "Interpretação dos resultados",
        subtitulo_style,
    )
)

if metricas_modelo:
    accuracy = metricas_modelo.get("accuracy")
    recall = metricas_modelo.get("recall_maligno")
    f1 = metricas_modelo.get("f1_maligno")
    roc_auc = metricas_modelo.get("roc_auc")

    if None not in [accuracy, recall, f1, roc_auc]:
        conteudo.append(
            texto_pdf(
                f"Na execução atual, o modelo selecionado foi {modelo_selecionado}. "
                f"A accuracy obtida foi {accuracy:.3f}, indicando a proporção geral "
                f"de classificações corretas no conjunto de teste. "
                f"O recall para malignidade foi {recall:.3f}, representando a "
                f"proporção de casos malignos identificados corretamente. "
                f"O F1-score foi {f1:.3f}, mostrando o equilíbrio entre precisão "
                f"e recall. A métrica ROC-AUC foi {roc_auc:.3f}, indicando a "
                f"capacidade de diferenciar padrões benignos e malignos."
            )
        )

        conteudo.append(
            texto_pdf(
                "Esses resultados representam desempenho técnico na base utilizada. "
                "Eles não representam validação clínica e não confirmam que o "
                "sistema terá o mesmo desempenho em outros hospitais, populações "
                "ou equipamentos."
            )
        )

else:
    conteudo.append(
        texto_pdf(
            "Não foi possível interpretar os resultados porque as métricas do "
            "modelo não estão disponíveis."
        )
    )

# Origem dos testes
conteudo.append(
    texto_pdf(
        "Origem dos testes",
        subtitulo_style,
    )
)

conteudo.append(texto_pdf(origem_teste))

# Gráficos
if graficos:
    conteudo.append(PageBreak())

    conteudo.append(
        texto_pdf(
            "Resultados obtidos: gráficos e análises",
            subtitulo_style,
        )
    )

    for titulo_grafico, caminho_grafico, explicacao in graficos:
        conteudo.append(texto_pdf(titulo_grafico, subtitulo_style))

        conteudo.append(
            Image(
                str(caminho_grafico),
                width=16 * cm,
                height=9 * cm,
            )
        )

        conteudo.append(texto_pdf(explicacao))

# Medidas que influenciaram a classificação
conteudo.append(
    texto_pdf(
        "Medidas que mais influenciaram a classificação",
        subtitulo_style,
    )
)

conteudo.append(
    texto_pdf(
        "Além de informar a probabilidade de uma amostra ser classificada como "
        "benigna ou maligna, o sistema analisa quais medidas tiveram maior "
        "influência no resultado. Entre elas podem aparecer raio, textura, "
        "área, perímetro, concavidade e simetria da massa mamária."
    )
)

conteudo.append(
    texto_pdf(
        "A importância por permutação mostra quais atributos são mais relevantes "
        "para o desempenho geral do modelo. O SHAP complementa essa análise ao "
        "mostrar como cada atributo pode aumentar ou reduzir a probabilidade "
        "estimada em uma previsão específica."
    )
)

conteudo.append(
    texto_pdf(
        "Essas informações ajudam a tornar o resultado mais compreensível para "
        "a análise humana. Entretanto, elas não comprovam que uma característica "
        "causou câncer e não substituem avaliação médica, exames de imagem ou biópsia."
    )
)

# Limites clínicos
conteudo.append(
    texto_pdf(
        "Interpretação e limites clínicos",
        subtitulo_style,
    )
)

conteudo.append(
    texto_pdf(
        "Esta análise é apoio educacional à triagem. O resultado não substitui "
        "consulta médica, mamografia, biópsia, histórico clínico ou protocolos "
        "assistenciais. Probabilidade baixa não exclui câncer e probabilidade alta "
        "não confirma câncer. A decisão final deve ser tomada por profissionais de saúde."
    )
)

# Geração do PDF
documento.build(
    conteudo,
    onFirstPage=adicionar_numero_pagina,
    onLaterPages=adicionar_numero_pagina,
)

print(f"PDF completo gerado em: {pdf_path}")